# 01 · Data audit, cleaning and feature engineering
**UE24CS352A – Machine Learning · Mini-Project (Problem #101)** · **Team:** *<Name 1 – SRN>*, *<Name 2 – SRN>*

**Owner:** *<Name 1>* · **Reads:** `data/raw/cfb_box_scores_2000_2010.csv` · **Writes:** `data/processed/team_season_features.csv`, `data/processed/bowl_matchups.csv`, `reports/figures/01–02`

Pipeline: **01 (this notebook)** → 02_models_and_cv → 03_final_model_and_demo. Reusable code lives in `src/features.py`.

## Problem statement
Every December, fans play the **Bowl Pick'em** game: for each bowl game they pick a winner and assign a *unique* confidence value from 1 to N
(N = number of bowl games). A correct pick earns its confidence points, and the highest total wins. So the model has to
(1) **pick the winner** of each bowl game from regular-season statistics and (2) **say how sure it is**.

**Approach.** Predict the **score margin** (team A − team B): its sign is the pick and its magnitude is the confidence. This builds on the Stanford CS229
project *"College Football Bowl Game Predictor"* (Cheshire, Childs, Leung), which reported ~66 % pick accuracy.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # lets us import the shared code in src/
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_PROC, FIG_DIR, FIRST_STATS_SEASON, MIN_GAMES, SEED, ensure_dirs
from src import features as F
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50, "display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
np.random.seed(SEED)
ensure_dirs()
print("Project root:", ROOT)

## 2. Data audit
The dataset has one row per game with final scores and (for some seasons) box-score statistics for the *away* and *home* team.
Before modelling we check what is actually usable.

In [ ]:
raw = F.load_raw()
print("Shape:", raw.shape)
print("Seasons:", raw.season.min(), "-", raw.season.max())
display(raw.game_type.value_counts().to_frame("games"))
display(raw.head(3))

In [ ]:
# How many games per season / type actually have box-score statistics?
raw["has_stats"] = raw["total_yards_away"].notna() & raw["total_yards_home"].notna()
audit = (raw.groupby(["season", "game_type"])["has_stats"]
            .agg(games="size", with_stats="sum").unstack("game_type").fillna(0).astype(int))
display(audit)

fig, ax = plt.subplots(figsize=(8, 3.2))
share = raw[raw.game_type == "regular"].groupby("season")["has_stats"].mean() * 100
ax.bar(share.index, share.values, color="#4C78A8")
ax.set_xlabel("Season"); ax.set_ylabel("% regular-season games with stats")
ax.set_title("Box-score coverage by season"); ax.set_xticks(share.index)
plt.tight_layout(); plt.savefig(FIG_DIR / "01_stats_coverage.png"); plt.show()

**Findings**
- Box-score stats are **entirely missing for 2000–2003**, so those seasons cannot be used for feature building. We use **2004–2010**.
- Bowl games are labelled `post` (the 48 `bowl_candidate` rows in 2000–2001 have no stats and are ignored).
- The file has no kickoff/punt returns, field goals or touchdown columns, so our feature set differs from the original paper.
- `possession_*` contains impossible values (e.g. 278 minutes), so we **exclude it**.

## 3. Cleaning

In [ ]:
reg, post = F.clean(raw)

## 4. Feature engineering
We convert each game to a **team-game** (long) table, with one row per team showing its own stats and its opponent's stats
(the opponent's offence is this team's *defence allowed*). Then we aggregate each team's **regular-season** games into one
feature vector per (season, team).

*Leakage guard:* features use only `game_type == "regular"`, so the bowl game itself never feeds its own prediction.

In [ ]:
team_games = F.to_team_games(reg)
team_feats = F.build_team_features(team_games)
print(f"{len(team_feats)} team-seasons, {len(F.FEATS)} features")
display(team_feats.loc[(2010, ["Auburn", "Oregon"]), F.FEATS].T.round(2)
        if (2010, "Auburn") in team_feats.index else team_feats.head(2)[F.FEATS].T.round(2))

## 5. Bowl matchup dataset
For every bowl game, `team_a` is the team listed as *home* and `team_b` the other. Because bowls are mostly neutral-site games, the
A/B labelling is arbitrary. We therefore model the **difference** `feature(A) − feature(B)` and use symmetric (no-intercept) models
so that swapping A and B exactly flips the prediction.

Target: `margin = score_A − score_B`. The winner label is `margin > 0`.

In [ ]:
bowls = F.build_bowl_matchups(post, team_feats)

bowls.to_csv(DATA_PROC / "bowl_matchups.csv", index=False)
team_feats.reset_index().to_csv(DATA_PROC / "team_season_features.csv", index=False)
print("Saved processed data to", DATA_PROC)

### Quick exploratory look
Which single differential features correlate most with the final margin?

In [ ]:
corr = bowls[F.D_COLS].corrwith(bowls["margin"]).sort_values()
top = pd.concat([corr.head(6), corr.tail(6)])
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh([c[2:] for c in top.index], top.values, color=np.where(top.values > 0, "#54A24B", "#E45756"))
ax.set_xlabel("Pearson r with bowl-game margin (A − B)"); ax.set_title("Strongest single-feature signals")
plt.tight_layout(); plt.savefig(FIG_DIR / "02_feature_correlations.png"); plt.show()

**Next:** open `02_models_and_cv.ipynb`.